# Non-2023 Hybrid Experiment

## Protocol

**Status:** fixed on 2026-10-16, before any model in this notebook was run. The protocol was written after notebooks 06–08 and is recorded in git before execution.

### 1. Framing

This is a **pre-specified sensitivity analysis restricted to non-2023 tissues**. It is **not** an independent confirmation.

The hypothesis comes from an exploratory result in `08_missing_information.ipynb` (Section 10.2): on non-2023 tissues, hybrid minus tabular macro F1 was +0.036 [−0.004, +0.075], p = 0.078. That result used the same tissues, folds, embeddings and classifier. This analysis differs in two respects only:

1. Models are **trained** on non-2023 tissues only (in notebook 08 they were trained on all years).
2. Year is defined from `Tissue_ID` (notebook 08 used the Konan exam year first, which placed 3 tissues with a 2023 ID in other years).

The question is whether the exploratory signal persists when training excludes the 2023 subset. A similar result will be reported as consistent with notebook 08, not as confirmation.

### 2. Research question

Do specular microscopy image features provide additional information for expert donor cornea grading beyond the available tabular clinical and morphological features, when evaluation is restricted to tissues from years other than 2023?

### 3. Cohort

| Criterion | Definition |
|---|---|
| Labelled | `Tissue_Grade_Clean` in {A+, A, B} (`ms.load_labelled_data()`) |
| Usable image | Present in `clean_image_manifest.csv`, excluding the two tissues flagged for manual review (`NEB-2024-08-16259 R`, `NEB-2025-09-17471 R`) |
| Year | Four-digit year in `Tissue_ID` (`NEB-YYYY-…`) |
| Eligible | Year ≠ 2023 |

**Expected cohort:** 551 tissues from 312 donors; A+ 277, A 196, B 78. By year: 2020: 128, 2021: 12, 2022: 58, 2024: 168, 2025: 185. The notebook asserts these counts and stops if any differ.

### 4. Features and models

| Element | Setting |
|---|---|
| Tabular features | The 24 features in `ms.ALL_FEATURES`, preprocessed with `ms.make_preprocessor()` |
| Image features | Cached `resnet50_imagenet_fullres.npz` (frozen ImageNet ResNet-50, full resolution, 2048-d); no re-extraction |
| Image reduction | StandardScaler → PCA, 16 components, `random_state=42` |
| Model A: Tabular | Tabular preprocessing → Random Forest |
| Model B: Hybrid | Tabular preprocessing + image reduction → Random Forest |
| Random Forest | `n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1, max_features="sqrt", class_weight="balanced", random_state=42` (identical for both models; no tuning) |

### 5. Validation

- Existing folds `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to the eligible cohort. Training and test tissues in every fold are non-2023 only.
- All preprocessing and PCA are fitted on the training fold only, through a cloned pipeline.
- Checks, which must pass before any model is fitted:
  - no donor appears in both the training and test sets of any fold;
  - every eligible tissue is tested exactly once per repeat;
  - every test fold contains at least one tissue of each grade.

### 6. Metrics

| Role | Metric | Level |
|---|---|---|
| **Primary** | Macro F1 | Per fold (25 paired values) |
| Secondary | Balanced accuracy, accuracy, grade B recall | Per fold |
| Descriptive | Per-grade precision and recall | Pooled out-of-fold predictions over all repeats |
| Descriptive | Repeat-level macro F1 | Pooled out-of-fold predictions per repeat |

### 7. Statistical analysis

- **Primary comparison:** hybrid minus tabular macro F1, paired by fold.
- Report the mean paired difference, a corrected 95% confidence interval and a corrected paired t-test (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004). The variance is inflated by n_test / n_train using mean fold sizes, with 24 degrees of freedom.
- Secondary metrics use the same test. They are reported unadjusted and labelled secondary; no claim rests on them.
- Repeat-level differences are descriptive. Repeats reuse the same tissues and are not treated as independent.

**Per-year results (descriptive):**

- Using pooled out-of-fold predictions over all repeats, report tabular macro F1, hybrid macro F1 and their difference for each year (2020, 2021, 2022, 2024, 2025), with tissue counts.
- Report the overall difference recomputed with each year excluded in turn, to show whether any single year drives the result.
- 2021 (12 tissues) is shown but flagged as too small to interpret. No significance tests are run per year.

### 8. Interpretation rules

Fixed before running; the conclusion is read from the primary comparison only.

| Primary result (macro F1 difference) | Reported as |
|---|---|
| 95% CI lower bound > 0 | Hybrid outperformed tabular in this sensitivity analysis |
| 95% CI includes 0 | Inconclusive: no evidence of a difference |
| 95% CI upper bound < 0 | Hybrid underperformed tabular |

**Additional rules:**

- **Grade B recall qualifier:** if the mean fold-level grade B recall difference is below −0.02, the result is reported together with the reduced B recall, and the hybrid is not recommended for use, whatever the macro F1 result. The −0.02 threshold is a project-defined decision rule fixed for this analysis, not a clinically validated safety threshold.
- **Per-year qualifier:** if excluding any single year turns a positive overall difference to zero or below, the result is reported as dependent on that year.
- **Framing:** in every case the result is described as a sensitivity analysis, not independent confirmation of notebook 08.
- **No changes after results:** no alternative PCA sizes, image models, classifiers, hyperparameters, cohorts or metrics. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is recorded in the deviations log with its reason.

### 9. Proposed future validation plan

This section is a proposed plan, not evidence that future data will be available or independent. It applies only if a genuinely new cohort is obtained.

If such a cohort becomes available, the same protocol will be applied unchanged to it. The cohort must consist of independently collected and graded tissues: those not present in `cleaned_grading_dataset.xlsx` at commit `6e9c460`, excluding any from 2023.

- Both models are fitted once on all eligible tissues from the current cohort and evaluated once on the new cohort.
- Uncertainty for the paired macro F1 difference comes from a donor-level bootstrap (2,000 resamples, seed 42).
- The interpretation rules in Section 8 apply.
- Only that test, on a cohort verified as new and independently graded, would count as confirmatory.

# Analysis

Implements the protocol above without changes. Section numbers refer to the protocol.

In [1]:
import hashlib
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

SEED = ms.RANDOM_STATE
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
REVIEW_IDS = ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]
PCA_COMPONENTS = 16

# Protocol Section 4.
RF_PARAMS = dict(
    n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1,
    max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=-1,
)

# Protocol Section 3.
EXPECTED_TISSUES = 551
EXPECTED_DONORS = 312
EXPECTED_GRADES = {"A+": 277, "A": 196, "B": 78}
EXPECTED_YEARS = {2020: 128, 2021: 12, 2022: 58, 2024: 168, 2025: 185}

# Provenance of the input data (not part of the protocol; recorded for audit).
for path in [ms.DATA_PATH, ms.DONOR_FOLDS_PATH, EMBEDDING_PATH]:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path.name}: sha256 {digest[:16]}…")

cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
donor_grouped_folds.csv: sha256 755280ff9d7bd388…
resnet50_imagenet_fullres.npz: sha256 883d4a1723a17be7…


## Cohort (Protocol Section 3)

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin({normalize_id(i) for i in REVIEW_IDS})]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
labelled["Year"] = labelled["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype("Int64")

with_image = labelled.merge(
    manifest[["Match_ID"]], on="Match_ID", how="inner", validate="one_to_one"
)
cohort = (
    with_image[with_image["Year"].notna() & (with_image["Year"] != 2023)]
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)

grades = cohort[ms.TARGET].value_counts().to_dict()
years = {int(k): v for k, v in cohort["Year"].value_counts().sort_index().items()}
print("Tissues:", len(cohort))
print("Donors:", cohort["Donor_Group"].nunique())
print("Grades:", grades)
print("Years:", years)

assert len(cohort) == EXPECTED_TISSUES, "Tissue count differs from protocol"
assert cohort["Donor_Group"].nunique() == EXPECTED_DONORS, "Donor count differs from protocol"
assert grades == EXPECTED_GRADES, "Grade counts differ from protocol"
assert years == EXPECTED_YEARS, "Year counts differ from protocol"
print("\nCohort matches the protocol.")

Tissues: 551
Donors: 312
Grades: {'A+': 277, 'A': 196, 'B': 78}
Years: {2020: np.int64(128), 2021: np.int64(12), 2022: np.int64(58), 2024: np.int64(168), 2025: np.int64(185)}

Cohort matches the protocol.


In [3]:
cached = np.load(EMBEDDING_PATH, allow_pickle=True)
row_of = {tissue_id: i for i, tissue_id in enumerate(cached["tissue_ids"])}
missing = [t for t in cohort["Tissue_ID"] if t not in row_of]
assert not missing, f"{len(missing)} cohort tissues have no cached embedding"
embeddings = cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]]

EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[ms.ALL_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()
print("Feature matrix:", X.shape)

Feature matrix: (551, 2072)


## Models (Protocol Section 4)

In [4]:
def make_arm(with_image):
    branches = [("tabular", ms.make_preprocessor(), ms.ALL_FEATURES)]
    if with_image:
        branches.append(("image", Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
        ]), EMBEDDING_COLUMNS))
    return Pipeline([
        ("features", ColumnTransformer(branches, remainder="drop")),
        ("classifier", RandomForestClassifier(**RF_PARAMS)),
    ])

ARMS = {"Tabular": make_arm(with_image=False), "Hybrid": make_arm(with_image=True)}

## Validation checks (Protocol Section 5)

All checks must pass before any model is fitted.

In [5]:
fold_splits = {
    repeat: list(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat))
    for repeat in range(ms.N_REPEATS)
}

for repeat, splits in fold_splits.items():
    tested = np.sort(np.concatenate([test for _, test in splits]))
    assert np.array_equal(tested, np.arange(len(cohort))), "Each tissue must be tested once per repeat"
    for train, test in splits:
        assert set(cohort.loc[train, "Donor_Group"]).isdisjoint(cohort.loc[test, "Donor_Group"]), "Donor leak"
        assert set(y[test]) == set(ms.CLASSES), "A test fold lacks a grade"

sizes = np.array([(len(tr), len(te)) for s in fold_splits.values() for tr, te in s])
N_TRAIN, N_TEST = sizes.mean(axis=0)
print("All validation checks passed.")
print(f"Fold sizes: train {sizes[:, 0].min()}–{sizes[:, 0].max()}, test {sizes[:, 1].min()}–{sizes[:, 1].max()}")

All validation checks passed.
Fold sizes: train 428–450, test 101–123


## Cross-validated evaluation (Protocol Sections 5–6)

In [6]:
METRICS = {
    "Macro F1": lambda t, p: f1_score(t, p, labels=ms.CLASSES, average="macro", zero_division=0),
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
    "Recall B": lambda t, p: recall_score(t, p, labels=["B"], average="macro", zero_division=0),
}

fold_rows = []
oof = {(arm, r): np.empty(len(cohort), dtype=object) for arm in ARMS for r in range(ms.N_REPEATS)}

start = time.time()
for repeat, splits in fold_splits.items():
    for fold, (train, test) in enumerate(splits):
        for arm, pipeline in ARMS.items():
            pred = clone(pipeline).fit(X.iloc[train], y[train]).predict(X.iloc[test])
            oof[(arm, repeat)][test] = pred
            fold_rows.append({
                "Arm": arm, "Repeat": repeat, "Fold": fold,
                **{m: fn(y[test], pred) for m, fn in METRICS.items()},
            })

fold_scores = pd.DataFrame(fold_rows)
print(f"{len(fold_scores)} fold evaluations in {(time.time() - start) / 60:.1f} min")
print(fold_scores.groupby("Arm")[list(METRICS)].agg(["mean", "std"]).reindex(ARMS).round(4).to_string())

50 fold evaluations in 0.6 min
        Macro F1         Balanced accuracy         Accuracy         Recall B        
            mean     std              mean     std     mean     std     mean     std
Arm                                                                                 
Tabular   0.7109  0.0441            0.7049  0.0454   0.7161  0.0373   0.7222  0.1204
Hybrid    0.7484  0.0411            0.7343  0.0460   0.7604  0.0360   0.7144  0.1238


## Primary and secondary comparisons (Protocol Section 7)

In [7]:
def corrected_ttest(differences):
    d = np.asarray(differences, dtype=float)
    n = len(d)
    se = np.sqrt((1 / n + N_TEST / N_TRAIN) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), d.mean() - half, d.mean() + half, t, p


wide = fold_scores.pivot_table(index=["Repeat", "Fold"], columns="Arm", values=list(METRICS))
comparison = []
for metric in METRICS:
    d = wide[(metric, "Hybrid")] - wide[(metric, "Tabular")]
    mean, low, high, t, p = corrected_ttest(d)
    comparison.append({
        "Role": "Primary" if metric == "Macro F1" else "Secondary",
        "Metric": metric,
        "Tabular": wide[(metric, "Tabular")].mean(),
        "Hybrid": wide[(metric, "Hybrid")].mean(),
        "Difference": mean, "CI low": low, "CI high": high,
        "t": t, "p": p, "Folds improved": f"{(d > 0).sum()}/{len(d)}",
    })
comparison = pd.DataFrame(comparison)
print("Hybrid minus Tabular (corrected repeated-CV t-test, df = 24; secondary p-values unadjusted):")
print(comparison.round(4).to_string(index=False))

Hybrid minus Tabular (corrected repeated-CV t-test, df = 24; secondary p-values unadjusted):
     Role            Metric  Tabular  Hybrid  Difference  CI low  CI high       t      p Folds improved
  Primary          Macro F1   0.7109  0.7484      0.0375 -0.0125   0.0876  1.5465 0.1351          19/25
Secondary Balanced accuracy   0.7049  0.7343      0.0294 -0.0183   0.0771  1.2726 0.2153          17/25
Secondary          Accuracy   0.7161  0.7604      0.0443 -0.0032   0.0918  1.9238 0.0663          20/25
Secondary          Recall B   0.7222  0.7144     -0.0078 -0.1046   0.0890 -0.1662 0.8694           7/25


In [8]:
repeat_level = pd.DataFrame([
    {"Repeat": r, **{arm: METRICS["Macro F1"](y, oof[(arm, r)]) for arm in ARMS}}
    for r in range(ms.N_REPEATS)
]).set_index("Repeat")
repeat_level["Difference"] = repeat_level["Hybrid"] - repeat_level["Tabular"]
print("Repeat-level macro F1 (descriptive; repeats reuse the same tissues):")
print(repeat_level.round(4).to_string())

Repeat-level macro F1 (descriptive; repeats reuse the same tissues):
        Tabular  Hybrid  Difference
Repeat                             
0        0.7012  0.7427      0.0415
1        0.7295  0.7570      0.0274
2        0.6993  0.7506      0.0513
3        0.7202  0.7605      0.0403
4        0.7209  0.7458      0.0249


## Per-grade results (Protocol Section 6, descriptive)

In [9]:
y_pooled = np.tile(y, ms.N_REPEATS)
pooled = {arm: np.concatenate([oof[(arm, r)] for r in range(ms.N_REPEATS)]) for arm in ARMS}

per_grade = pd.concat({
    arm: pd.DataFrame({
        "Precision": precision_score(y_pooled, pred, labels=ms.CLASSES, average=None, zero_division=0),
        "Recall": recall_score(y_pooled, pred, labels=ms.CLASSES, average=None, zero_division=0),
    }, index=ms.CLASSES)
    for arm, pred in pooled.items()
}, axis=1)
print("Pooled over all repeats:")
print(per_grade.round(3).to_string())

Pooled over all repeats:
     Tabular           Hybrid       
   Precision Recall Precision Recall
A+     0.751  0.803     0.771  0.878
A      0.633  0.591     0.706  0.612
B      0.788  0.726     0.851  0.715


## Per-year results (Protocol Section 7, descriptive)

No significance tests are run per year. 2021 has 12 tissues and is too small to interpret.

In [10]:
year_pooled = np.tile(cohort["Year"].to_numpy(dtype=int), ms.N_REPEATS)
macro_f1 = METRICS["Macro F1"]

by_year = []
for year in sorted(EXPECTED_YEARS):
    sel = year_pooled == year
    tab, hyb = macro_f1(y_pooled[sel], pooled["Tabular"][sel]), macro_f1(y_pooled[sel], pooled["Hybrid"][sel])
    by_year.append({
        "Year": year, "Tissues": EXPECTED_YEARS[year],
        "Tabular": tab, "Hybrid": hyb, "Difference": hyb - tab,
        "Note": "too small to interpret" if year == 2021 else "",
    })
print("Macro F1 by year (pooled over repeats):")
print(pd.DataFrame(by_year).round(4).to_string(index=False))

overall_pooled = macro_f1(y_pooled, pooled["Hybrid"]) - macro_f1(y_pooled, pooled["Tabular"])
leave_out = []
for year in sorted(EXPECTED_YEARS):
    keep = year_pooled != year
    leave_out.append({
        "Excluded year": year,
        "Difference": macro_f1(y_pooled[keep], pooled["Hybrid"][keep])
                      - macro_f1(y_pooled[keep], pooled["Tabular"][keep]),
    })
leave_out = pd.DataFrame(leave_out)
print(f"\nOverall pooled difference: {overall_pooled:+.4f}")
print("Difference with each year excluded:")
print(leave_out.round(4).to_string(index=False))

Macro F1 by year (pooled over repeats):


 Year  Tissues  Tabular  Hybrid  Difference                   Note
 2020      128   0.6669  0.7097      0.0428                       
 2021       12   0.7130  0.7587      0.0457 too small to interpret
 2022       58   0.7677  0.7728      0.0051                       
 2024      168   0.7004  0.7394      0.0391                       
 2025      185   0.7243  0.7615      0.0372                       



Overall pooled difference: +0.0371
Difference with each year excluded:
 Excluded year  Difference
          2020      0.0344
          2021      0.0362
          2022      0.0406
          2024      0.0369
          2025      0.0363


## Interpretation (Protocol Section 8)

The rules are applied mechanically to the results above.

In [11]:
primary = comparison.set_index("Metric").loc["Macro F1"]
b_recall_difference = comparison.set_index("Metric").loc["Recall B", "Difference"]

if primary["CI low"] > 0:
    verdict = "Hybrid outperformed tabular in this sensitivity analysis"
elif primary["CI high"] < 0:
    verdict = "Hybrid underperformed tabular"
else:
    verdict = "Inconclusive: no evidence of a difference"

b_qualifier = b_recall_difference < -0.02
year_dependent = overall_pooled > 0 and (leave_out["Difference"] <= 0).any()

print(f"Primary: macro F1 difference {primary['Difference']:+.4f} "
      f"[{primary['CI low']:+.4f}, {primary['CI high']:+.4f}], p = {primary['p']:.4f}")
print(f"Verdict: {verdict}")
print(f"Grade B recall qualifier triggered: {b_qualifier} "
      f"(mean B recall difference {b_recall_difference:+.4f}; threshold −0.02)")
if b_qualifier:
    print("  -> Reported with reduced grade B recall; hybrid not recommended for use.")
print(f"Per-year qualifier triggered: {year_dependent}")
if year_dependent:
    years_flagged = leave_out.loc[leave_out["Difference"] <= 0, "Excluded year"].tolist()
    print(f"  -> Result depends on year(s): {years_flagged}")
print("Framing: sensitivity analysis restricted to non-2023 tissues; not independent confirmation of notebook 08.")

Primary: macro F1 difference +0.0375 [-0.0125, +0.0876], p = 0.1351
Verdict: Inconclusive: no evidence of a difference
Grade B recall qualifier triggered: False (mean B recall difference -0.0078; threshold −0.02)
Per-year qualifier triggered: False
Framing: sensitivity analysis restricted to non-2023 tissues; not independent confirmation of notebook 08.


In [12]:
fold_scores.to_csv(ms.PROCESSED_DIR / "non_2023_fold_scores.csv", index=False)
pd.DataFrame({
    "Tissue_ID": np.tile(cohort["Tissue_ID"], ms.N_REPEATS),
    "Year": year_pooled,
    "Repeat": np.repeat(range(ms.N_REPEATS), len(cohort)),
    "Actual": y_pooled,
    **pooled,
}).to_csv(ms.PROCESSED_DIR / "non_2023_oof_predictions.csv", index=False)
print("Saved fold scores and out-of-fold predictions to", ms.PROCESSED_DIR)

Saved fold scores and out-of-fold predictions to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed


## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| — | None | — |